# Fine-tune DistilBERT on SDCNL (Google Colab)
**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all (about 5-10 minutes).

In [ ]:
!pip -q install transformers datasets accelerate
!git clone -q https://github.com/ayaanzhaque/SDCNL

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)

SEED = 42
set_seed(SEED)

# --- same data and same split as the laptop scripts (src/common.py) ---
def build(df):
    out = pd.DataFrame()
    out["raw_text"] = (df["title"].fillna("") + ". " + df["selftext"].fillna("")).str.strip()
    out["label"] = df["is_suicide"].astype(int)
    return out

train_full = build(pd.read_csv("SDCNL/data/training-set.csv"))
test = build(pd.read_csv("SDCNL/data/testing-set.csv")).reset_index(drop=True)
train, val = train_test_split(train_full, test_size=0.15, stratify=train_full["label"], random_state=SEED)
print(len(train), len(val), len(test))   # must print 1288 228 379

## Tokenise
BERT-style models cannot read raw text; the tokenizer cuts it into word-pieces and turns them into numbers.

In [ ]:
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(df):
    ds = Dataset.from_dict({"text": df["raw_text"].tolist(), "label": df["label"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

ds_train, ds_val, ds_test = to_ds(train), to_ds(val), to_ds(test)

## Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary")
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

args = TrainingArguments(
    output_dir="distilbert_out", num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", seed=SEED, report_to="none", fp16=torch.cuda.is_available())

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  processing_class=tok, compute_metrics=metrics)
trainer.train()

## Evaluate on the official test set and save predictions

In [ ]:
out = trainer.predict(ds_test)
logits = out.predictions
prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
pred = logits.argmax(-1)
y = test["label"].values
pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary")
print(f"TEST  accuracy={accuracy_score(y, pred):.4f}  precision={pr:.4f}  recall={rc:.4f}  "
      f"F1={f1:.4f}  ROC-AUC={roc_auc_score(y, prob):.4f}")

pd.DataFrame({"DistilBERT": pred, "DistilBERT_prob": prob, "y_true": y}).to_csv(
    "distilbert_test_predictions.csv", index=False)

## Download
Put the downloaded file into your project's `outputs/` folder, then re-run `python src/03_evaluate_xai.py`.

In [ ]:
from google.colab import files
files.download("distilbert_test_predictions.csv")